# CineScore V6.2 Data Engineering: TMDB API Franchise Enrichment
The backend database omission of `belongs_to_collection` broke our Franchise Target flag. This script bridges that gap by acting as an Enterprise REST API wrapper. It iterates over the cleaned 3,350+ movies, dynamically queries the TMDB API, extracts the Collection Object, rate-limits identically to TMDB's 40 req/sec ceiling, and checkpoints the new data locally.

In [ ]:
%pip install -q requests tqdm python-dotenv


In [ ]:
import pandas as pd
import requests
import time
import os
from tqdm import tqdm
from dotenv import load_dotenv

# File Paths Logic (Google Colab vs Local)
if os.path.exists('/content'):
    print("Detected Google Colab Environment")
    INPUT_PATH = "/content/Data/Processed_Dataset/master_analytical_df.csv"
    OUTPUT_PATH = "/content/Data/Processed_Dataset/v6_master_analytical_df.csv"
else:
    print("Detected Local Environment")
    INPUT_PATH = "../Data/Processed_Dataset/master_analytical_df.csv"
    OUTPUT_PATH = "../Data/Processed_Dataset/v6_master_analytical_df.csv"


### Step 1: Initialize the Engine & Authenticate API

In [ ]:
# Load private .env variables automatically
load_dotenv()

# Note: Change 'TMDB_API_KEY' if your variable in the .env is named differently (e.g. 'TMDB_KEY')
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    print("\n⚠️ CRITICAL WARNING: API Key not found! Is your .env file in the same directory, and is the variable named 'TMDB_API_KEY'?")

print(f"Loading Cleaned Dataset: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {df.shape[0]} rows ready for API Enrichment.")


### Step 2: Live Ping & Rate-Limited Franchise Extraction

In [ ]:
def fetch_franchise_status(movie_id, api_key):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            collection = data.get('belongs_to_collection', None)
            if collection is not None and isinstance(collection, dict):
                return 1
            return 0
        elif response.status_code == 429:
            # Hard Rate-Limit Wait
            time.sleep(2)
            return fetch_franchise_status(movie_id, api_key)
        else:
            return 0
    except Exception as e:
        return 0

# Let's create an array so the dataframe math operates cleanly
franchise_flags = []

print("Starting TMDB API ping sweep...")
# tqdm tracks iteration speed and estimated time of arrival (ETA)
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Enriching Franchises"):
    movie_id = row['id']
    
    # Hit the API
    is_franchise = fetch_franchise_status(movie_id, TMDB_API_KEY)
    franchise_flags.append(is_franchise)
    
    # TMDB rate limits to ~40-50 requests per second. 
    # 0.03sec sleep mathematically caps us at exactly exactly ~33 requests per second to ensure zero IP bans.
    time.sleep(0.03)
    
df['is_franchise'] = franchise_flags


### Step 3: Checkpointing

In [ ]:
franchise_count = df['is_franchise'].sum()
print(f"\nEnrichment Complete. Found {franchise_count} confirmed franchise/collection movies.")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Enterprise Checkpoint Saved explicitly to {OUTPUT_PATH}. You do not need to run this API fetch ever again for this cut.")
